# <font color="#418FDE" size="6.5" uppercase>**Scaling Workloads**</font>

>Last update: 20260714.
    
By the end of this Lecture, you will be able to:
- Design out-of-core workflows using batches and incremental estimators. 
- Use hashing, sparse matrices, and incremental preprocessing to reduce memory pressure. 
- Measure prediction latency and throughput for fitted models. 


## **1. Out of Core**

### **1.1. Streaming Data Batches**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_31/Lecture_A/image_01_01.jpg?v=1784041870" width="250">



>* Process huge datasets in manageable batches
>* Balance batch size for memory efficiency

>* Read, transform, and release each batch
>* Lower memory use and restart reliably

>* Tune batch size for memory and speed
>* Process consistently with safeguards and monitoring



In [ ]:
#@title Python Code - Streaming Data Batches

# Stream batches through an incremental classifier.
# Each batch updates the model once.
# The output compares memory-friendly batch progress.

import numpy as np
import sklearn
from sklearn.datasets import make_classification
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

# Create a small deterministic dataset for streaming practice.
features, labels = make_classification(
    n_samples=3000,
    n_features=12,
    n_informative=8,
    n_redundant=2,
    random_state=42,
)

# Keep a final test set separate from streamed training batches.
train_features, test_features, train_labels, test_labels = train_test_split(
    features,
    labels,
    test_size=0.25,
    stratify=labels,
    random_state=42,
)

# Validate the basic shape before streaming begins.
if train_features.shape[0] != train_labels.shape[0]:
    raise ValueError("Training features and labels must align.")

batch_size = 250
classes = np.array([0, 1])
model = SGDClassifier(loss="log_loss", max_iter=1, tol=None, random_state=42)

# Feed one manageable batch at a time.
batch_count = 0
for start in range(0, train_features.shape[0], batch_size):
    stop = min(start + batch_size, train_features.shape[0])
    batch_features = train_features[start:stop]
    batch_labels = train_labels[start:stop]

    model.partial_fit(batch_features, batch_labels, classes=classes)
    batch_count = batch_count + 1

predictions = model.predict(test_features)
accuracy = accuracy_score(test_labels, predictions)

print("scikit-learn version:", sklearn.__version__)
print("Training rows streamed:", train_features.shape[0])
print("Batch size:", batch_size)
print("Number of batches:", batch_count)
print("Largest batch shape:", (batch_size, train_features.shape[1]))
print("Held-out accuracy:", round(accuracy, 3))



### **1.2. Incremental Model Updates**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_31/Lecture_A/image_01_02.jpg?v=1784041868" width="250">



>* Train on manageable batches sequentially
>* Model parameters retain learning across updates

>* Shuffle representative batches to reduce bias
>* Tune batch size and validate regularly

>* Save model state for safe recovery
>* Update only when streams remain reliable



In [ ]:
#@title Python Code - Incremental Model Updates

# This example trains a model in batches.
# Incremental updates avoid loading all training data.
# Accuracy after each batch shows gradual learning.

import numpy as np
import sklearn
from sklearn.datasets import make_classification
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Create a small deterministic classification dataset.
features, labels = make_classification(
    n_samples=3000,
    n_features=12,
    n_informative=8,
    n_redundant=2,
    random_state=42,
)

# Split once so testing data stays unseen.
train_features, test_features, train_labels, test_labels = train_test_split(
    features,
    labels,
    test_size=0.25,
    stratify=labels,
    random_state=42,
)

# Validate the basic shape before training.
if train_features.shape[0] != train_labels.shape[0]:
    raise ValueError("Training rows and labels must match.")

# Shuffle training rows to make batches more representative.
rng = np.random.default_rng(42)
order = rng.permutation(train_features.shape[0])
train_features = train_features[order]
train_labels = train_labels[order]

# Use incremental preprocessing and an incremental classifier.
scaler = StandardScaler()
model = SGDClassifier(loss="log_loss", max_iter=1, tol=None, random_state=42)
classes = np.array([0, 1])

# Update the scaler and model one batch at a time.
batch_size = 300
batch_numbers = []
accuracies = []

for start in range(0, train_features.shape[0], batch_size):
    stop = min(start + batch_size, train_features.shape[0])
    batch_features = train_features[start:stop]
    batch_labels = train_labels[start:stop]

    scaler.partial_fit(batch_features)
    scaled_batch = scaler.transform(batch_features)
    model.partial_fit(scaled_batch, batch_labels, classes=classes)

    scaled_test = scaler.transform(test_features)
    predictions = model.predict(scaled_test)
    accuracy = accuracy_score(test_labels, predictions)

    batch_numbers.append(len(batch_numbers) + 1)
    accuracies.append(accuracy)

# Print concise results from the incremental workflow.
print("scikit-learn version:", sklearn.__version__)
print("Batches processed:", len(batch_numbers))
print("Final test accuracy:", round(accuracies[-1], 3))
print("Accuracy after first three batches:", [round(x, 3) for x in accuracies[:3]])
print("Accuracy after last three batches:", [round(x, 3) for x in accuracies[-3:]])



### **1.3. Mini Batch Training**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_31/Lecture_A/image_01_03.jpg?v=1784041872" width="250">



>* Train large datasets in manageable batches
>* Update models while keeping memory predictable

>* Batch size affects memory, speed, and stability
>* Balance hardware efficiency with reliable learning

>* Build batches to reduce ordering bias
>* Monitor learning, shifts, and safe recovery



In [ ]:
#@title Python Code - Mini Batch Training

# Mini batches train models without loading everything.
# Incremental estimators update from each small batch.
# Accuracy history shows learning across repeated batches.

import numpy as np
import sklearn
from sklearn.datasets import make_classification
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Create a small stand-in for a much larger dataset.
features, labels = make_classification(
    n_samples=4000, n_features=12, n_informative=8, random_state=42
)

# Split once so evaluation data stays unseen during training.
train_features, test_features, train_labels, test_labels = train_test_split(
    features, labels, test_size=0.25, stratify=labels, random_state=42
)

# Validate the basic shape before starting incremental work.
if train_features.shape[0] != train_labels.shape[0]:
    raise ValueError("Training rows and labels must match.")

# Incremental preprocessing learns scaling statistics batch by batch.
batch_size = 250
scaler = StandardScaler()

for start in range(0, train_features.shape[0], batch_size):
    stop = start + batch_size
    scaler.partial_fit(train_features[start:stop])

# The model supports partial_fit for mini batch updates.
model = SGDClassifier(loss="log_loss", max_iter=1, tol=None, random_state=42)
classes = np.unique(train_labels)
accuracy_history = []

# Each pass reads only one mini batch at a time.
for start in range(0, train_features.shape[0], batch_size):
    stop = start + batch_size
    batch_features = scaler.transform(train_features[start:stop])
    batch_labels = train_labels[start:stop]
    model.partial_fit(batch_features, batch_labels, classes=classes)
    test_scaled = scaler.transform(test_features)
    predictions = model.predict(test_scaled)
    accuracy_history.append(accuracy_score(test_labels, predictions))

# Print concise results that connect batches to learning progress.
print("scikit-learn version:", sklearn.__version__)
print("Mini batch size:", batch_size)
print("Number of mini batches:", len(accuracy_history))
print("Accuracy after first batch:", round(accuracy_history[0], 3))
print("Accuracy after final batch:", round(accuracy_history[-1], 3))



## **2. Memory Efficient Features**

### **2.1. Feature Hashing**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_31/Lecture_A/image_02_01.jpg?v=1784041861" width="250">



>* Maps huge feature sets into fixed columns
>* Avoids growing dictionaries for scalable processing

>* Fixed memory for unknown, growing features
>* Consistent batch transforms without stored vocabularies

>* Collisions merge features and reduce distinction
>* Larger sparse hashes balance memory and accuracy



In [ ]:
#@title Python Code - Feature Hashing

# Demonstrate feature hashing for compact sparse features.
# Hashing avoids storing a growing vocabulary dictionary.
# Output compares hashed and vocabulary based representations.

import numpy as np
import sklearn
from sklearn.feature_extraction import FeatureHasher
from sklearn.feature_extraction import DictVectorizer

# Each row is a tiny event with categorical feature names.
events = [
    {"word=refund": 1, "merchant=bookshop": 1, "device=mobile": 1},
    {"word=login": 1, "merchant=cafe": 1, "device=laptop": 1},
    {"word=refund": 1, "merchant=cafe": 1, "device=mobile": 1},
]

# DictVectorizer learns and stores every observed feature name.
vectorizer = DictVectorizer(sparse=True)
vocabulary_matrix = vectorizer.fit_transform(events)

# FeatureHasher uses a fixed number of columns without fitting.
hasher = FeatureHasher(n_features=8, input_type="dict", alternate_sign=False)
hashed_matrix = hasher.transform(events)

# Validate that both methods kept one row per event.
if vocabulary_matrix.shape[0] != len(events):
    raise ValueError("Vocabulary matrix row count is unexpected.")

if hashed_matrix.shape[0] != len(events):
    raise ValueError("Hashed matrix row count is unexpected.")

# Count how many original names land in each hashed column.
feature_names = vectorizer.get_feature_names_out()
feature_dicts = [{name: 1} for name in feature_names]
feature_columns = hasher.transform(feature_dicts).nonzero()[1]

# A collision means multiple names share one hashed column.
unique_columns = np.unique(feature_columns)
collision_count = len(feature_names) - len(unique_columns)

print("scikit-learn version:", sklearn.__version__)
print("Vocabulary shape:", vocabulary_matrix.shape)
print("Hashed shape:", hashed_matrix.shape)
print("Stored vocabulary names:", len(feature_names))
print("Hasher stored vocabulary names: 0")
print("Nonzero values in hashed matrix:", hashed_matrix.nnz)
print("Collisions with 8 hashed columns:", collision_count)



### **2.2. Incremental Preprocessing**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_31/Lecture_A/image_02_02.jpg?v=1784041864" width="250">



>* Process large datasets in small batches
>* Stream transformations to reduce memory use

>* Update feature summaries batch by batch
>* Avoid full-data copies and transformations

>* Keep batch transformations consistent over time
>* Balance usefulness, memory, stability, and reliability



In [ ]:
#@title Python Code - Incremental Preprocessing

# This example scales data in small batches.
# Incremental preprocessing avoids full dataset copies.
# The output compares memory and scaling results.

import numpy as np
import sklearn
from sklearn.preprocessing import StandardScaler

# A generator simulates rows arriving from storage.
rng = np.random.default_rng(42)
rows = 6000
features = 4
batch_size = 1000

# Small batches keep only part of the data active.
batches = []
for start in range(0, rows, batch_size):
    batch = rng.normal(loc=50, scale=10, size=(batch_size, features))
    batches.append(batch.astype(np.float32))

# The scaler learns running means and variances incrementally.
incremental_scaler = StandardScaler()
for batch in batches:
    incremental_scaler.partial_fit(batch)

# A full scaler is used only for comparison here.
full_data = np.vstack(batches)
full_scaler = StandardScaler()
full_scaler.fit(full_data)

# Transform one new batch using the learned global statistics.
new_batch = batches[0]
scaled_batch = incremental_scaler.transform(new_batch)

# Estimate memory pressure for full data versus one batch.
full_memory_mb = full_data.nbytes / 1_000_000
batch_memory_mb = new_batch.nbytes / 1_000_000
mean_difference = np.max(np.abs(incremental_scaler.mean_ - full_scaler.mean_))

# Validate that the incremental scaler learned compatible statistics.
if scaled_batch.shape != new_batch.shape:
    raise ValueError("Scaled data should keep the original shape.")

print(f"scikit-learn version: {sklearn.__version__}")
print(f"Rows processed: {rows} in batches of {batch_size}")
print(f"Full array memory: {full_memory_mb:.3f} MB")
print(f"One batch memory: {batch_memory_mb:.3f} MB")
print(f"Largest mean difference versus full fit: {mean_difference:.8f}")
print(f"Scaled first-row mean: {scaled_batch[0].mean():.3f}")



### **2.3. Sparse Feature Storage**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_31/Lecture_A/image_02_03.jpg?v=1784041866" width="250">



>* Many datasets contain mostly inactive features
>* Store only nonzeros to save memory

>* Store only active feature values
>* Fit larger datasets in memory

>* Choose algorithms that preserve sparse inputs
>* Avoid preprocessing that densifies feature matrices



In [ ]:
#@title Python Code - Sparse Feature Storage

# Compare dense and sparse feature storage.
# Sparse matrices keep only active values.
# Memory savings become visible with many zeros.

import numpy as np
from sklearn.feature_extraction import FeatureHasher

# Each row has only a few active categorical features.
records = [
    {"city=Boston": 1, "device=mobile": 1, "ad=sports": 1},
    {"city=Denver": 1, "device=desktop": 1, "ad=travel": 1},
    {"city=Boston": 1, "device=tablet": 1, "ad=music": 1},
]

# Hashing creates many possible columns without storing names.
hasher = FeatureHasher(n_features=1024, input_type="dict")
sparse_features = hasher.transform(records)

# Dense storage reserves space for every zero entry.
dense_features = sparse_features.toarray()

# Validate the example shape before comparing memory.
if sparse_features.shape != dense_features.shape:
    raise ValueError("Sparse and dense shapes should match.")

# Estimate memory used by each representation.
sparse_bytes = (
    sparse_features.data.nbytes
    + sparse_features.indices.nbytes
    + sparse_features.indptr.nbytes
)
dense_bytes = dense_features.nbytes

# Count how much of the matrix is actually active.
total_cells = sparse_features.shape[0] * sparse_features.shape[1]
active_cells = sparse_features.nnz
active_percent = 100 * active_cells / total_cells

print("Sparse feature storage with hashed categorical indicators")
print(f"Matrix shape: {sparse_features.shape[0]} rows x {sparse_features.shape[1]} columns")
print(f"Active nonzero cells: {active_cells} of {total_cells}")
print(f"Active share: {active_percent:.2f}%")
print(f"Dense memory: {dense_bytes:,} bytes")
print(f"Sparse memory: {sparse_bytes:,} bytes")
print(f"Dense uses about {dense_bytes / sparse_bytes:.1f} times more memory here.")



## **3. Prediction Performance**

### **3.1. Memory Mapped Models**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_31/Lecture_A/image_03_01.jpg?v=1784041874" width="250">



>* Map large models without fully loading RAM
>* Share read-only pages across prediction workers

>* Repeated access warms cache and speeds predictions
>* Scattered access can hurt tail latency

>* Serve immutable model files read-only
>* Measure cold, warm, and multi-worker performance



In [ ]:
#@title Python Code - Memory Mapped Models

# This example compares normal and memory-mapped prediction.
# Memory mapping can share large fitted arrays.
# We measure latency and throughput after loading.

import numpy as np
import sklearn
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression

# Create a small deterministic classification dataset.
features, target = make_classification(
    n_samples=3000,
    n_features=20,
    random_state=42,
)

# Fit one simple model for prediction timing.
model = LogisticRegression(max_iter=300, random_state=42)
model.fit(features, target)

# Store coefficients in a memory-mapped array without explicit files.
coef = model.coef_.astype(np.float64)
coef_map = np.memmap(
    "mapped_coefficients.dat",
    dtype=np.float64,
    mode="w+",
    shape=coef.shape,
)

# Copy fitted parameters into the mapped array.
coef_map[:] = coef[:]
coef_map.flush()

# Reopen the same array in read-only mapped mode.
mapped_coef = np.memmap(
    "mapped_coefficients.dat",
    dtype=np.float64,
    mode="r",
    shape=coef.shape,
)

# Validate that mapped and ordinary coefficients match.
if not np.allclose(mapped_coef, coef):
    raise ValueError("Mapped coefficients do not match the fitted model.")

# Use NumPy timing to avoid importing standard-library timing modules.
batch = features[:500].astype(np.float64)
repeats = 80

# Measure repeated prediction scores with ordinary coefficients.
start = np.datetime64("now", "ns")
for _ in range(repeats):
    normal_scores = batch @ coef.T + model.intercept_
normal_elapsed = np.datetime64("now", "ns") - start

# Measure repeated prediction scores with memory-mapped coefficients.
start = np.datetime64("now", "ns")
for _ in range(repeats):
    mapped_scores = batch @ mapped_coef.T + model.intercept_
mapped_elapsed = np.datetime64("now", "ns") - start

# Convert elapsed nanoseconds into beginner-friendly metrics.
normal_seconds = normal_elapsed.astype("timedelta64[ns]").astype(float) / 1e9
mapped_seconds = mapped_elapsed.astype("timedelta64[ns]").astype(float) / 1e9
predictions = batch.shape[0] * repeats

normal_latency_ms = 1000 * normal_seconds / repeats
mapped_latency_ms = 1000 * mapped_seconds / repeats
normal_throughput = predictions / normal_seconds
mapped_throughput = predictions / mapped_seconds

# Confirm both approaches produce the same predictions.
normal_labels = (normal_scores.ravel() >= 0).astype(int)
mapped_labels = (mapped_scores.ravel() >= 0).astype(int)
if not np.array_equal(normal_labels, mapped_labels):
    raise ValueError("Mapped predictions differ from normal predictions.")

print("scikit-learn version:", sklearn.__version__)
print("Batch size:", batch.shape[0])
print("Normal latency per batch, ms:", round(normal_latency_ms, 4))
print("Mapped latency per batch, ms:", round(mapped_latency_ms, 4))
print("Normal throughput, predictions/sec:", round(normal_throughput, 0))
print("Mapped throughput, predictions/sec:", round(mapped_throughput, 0))
print("Predictions match for normal and mapped coefficients:", True)



### **3.2. Latency Measurement**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_31/Lecture_A/image_03_02.jpg?v=1784041878" width="250">



>* Latency measures end-to-end prediction response time
>* Preprocessing can make production predictions slower

>* Match tests to real usage patterns
>* Compare cold, warm, and percentile latency

>* Match latency targets to application needs
>* Optimize and benchmark on deployment-like hardware



In [ ]:
#@title Python Code - Latency Measurement

# Measure prediction latency for a fitted model.
# Compare single requests with small batches.
# Report milliseconds and predictions per second.

import numpy as np
import sklearn
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Load a small packaged classification dataset.
data = load_breast_cancer()
features = data.data
target = data.target

# Check that features and labels match.
if features.shape[0] != target.shape[0]:
    raise ValueError("Feature rows must match target labels.")

# Split data before fitting the preprocessing pipeline.
X_train, X_test, y_train, y_test = train_test_split(
    features, target, test_size=0.25, stratify=target, random_state=42
)

# Fit one simple model with preprocessing included.
model = make_pipeline(
    StandardScaler(), LogisticRegression(max_iter=1000, random_state=42)
)
model.fit(X_train, y_train)

# Warm up the fitted pipeline before steady latency timing.
single_request = X_test[:1]
batch_request = X_test[:32]
model.predict(single_request)
model.predict(batch_request)

# Use NumPy timing to avoid extra imports.
repeat_count = 80
single_latencies_ms = []

# Time repeated one-row predictions.
for _ in range(repeat_count):
    start_time = np.datetime64("now", "ns")
    model.predict(single_request)
    end_time = np.datetime64("now", "ns")
    elapsed_ms = (end_time - start_time) / np.timedelta64(1, "ms")
    single_latencies_ms.append(float(elapsed_ms))

# Time repeated small-batch predictions.
batch_latencies_ms = []
for _ in range(repeat_count):
    start_time = np.datetime64("now", "ns")
    model.predict(batch_request)
    end_time = np.datetime64("now", "ns")
    elapsed_ms = (end_time - start_time) / np.timedelta64(1, "ms")
    batch_latencies_ms.append(float(elapsed_ms))

# Summarize typical and slower response times.
single_median = np.percentile(single_latencies_ms, 50)
single_p95 = np.percentile(single_latencies_ms, 95)
batch_median = np.percentile(batch_latencies_ms, 50)

# Convert batch latency into throughput.
batch_size = batch_request.shape[0]
throughput = batch_size / (batch_median / 1000)

print(f"scikit-learn version: {sklearn.__version__}")
print(f"Single prediction median latency: {single_median:.3f} ms")
print(f"Single prediction 95th percentile latency: {single_p95:.3f} ms")
print(f"Batch of {batch_size} median latency: {batch_median:.3f} ms")
print(f"Batch throughput at median latency: {throughput:.1f} predictions/s")



### **3.3. Throughput Measurement**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_31/Lecture_A/image_03_03.jpg?v=1784041876" width="250">



>* Measure predictions completed over time
>* Include full workflow steps in testing

>* Batch predictions can greatly increase throughput
>* Choose batch sizes that balance resource tradeoffs

>* Test throughput in realistic deployment conditions
>* Compare warm and cold performance



In [ ]:
#@title Python Code - Throughput Measurement

# Measure prediction throughput for fitted models.
# Batch size changes predictions per second.
# Results show faster batch processing rates.

import numpy as np
import sklearn
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Create a small deterministic classification dataset.
features, target = make_classification(
    n_samples=6000,
    n_features=20,
    random_state=42,
)

# Split data so preprocessing learns only from training rows.
train_features, test_features, train_target, test_target = train_test_split(
    features, target, test_size=0.4, stratify=target, random_state=42
)

# Scale features before fitting the simple classifier.
scaler = StandardScaler()
train_scaled = scaler.fit_transform(train_features)
test_scaled = scaler.transform(test_features)

# Fit one beginner-friendly model for prediction timing.
model = LogisticRegression(max_iter=300, random_state=42)
model.fit(train_scaled, train_target)

# Validate that the test set is large enough.
if len(test_scaled) < 1000:
    raise ValueError("Need at least 1000 test rows for this example.")

# Warm up the fitted model before measuring steady throughput.
model.predict(test_scaled[:100])

# Use NumPy timing to avoid extra imports.
batch_sizes = [1, 10, 100, 1000]
print("scikit-learn version:", sklearn.__version__)
print("Batch size | predictions per second")

# Measure repeated predictions for each batch size.
for batch_size in batch_sizes:
    batch = test_scaled[:batch_size]
    repeats = max(20, 2000 // batch_size)
    start_time = np.datetime64("now", "ns")
    for repeat_index in range(repeats):
        predictions = model.predict(batch)
    end_time = np.datetime64("now", "ns")
    elapsed_ns = (end_time - start_time) / np.timedelta64(1, "ns")
    throughput = batch_size * repeats / (elapsed_ns / 1_000_000_000)
    print(batch_size, "|", round(float(throughput), 0))

# Confirm the final batch produced the expected number of labels.
print("Last batch predictions:", len(predictions))



# <font color="#418FDE" size="6.5" uppercase>**Scaling Workloads**</font>


In this lecture, you learned to:
- Design out-of-core workflows using batches and incremental estimators. 
- Use hashing, sparse matrices, and incremental preprocessing to reduce memory pressure. 
- Measure prediction latency and throughput for fitted models. 

In the next Lecture (Lecture B), we will go over 'Parallel Configuration'